# This notebook shows how to query eigenstates and wavefunction overlaps at arbitrary k-points of bulk silicon
### Import the necessary libraries

In [1]:
import numpy as np

from elkpy.structure import Structure

SI_AVEC = [(5.13, 5.13, 0.00), (5.13, 0.00, 5.13), (0.00, 5.13, 5.13)]
SI_SPECIES = {"Si": [(0.0, 0.0, 0.0), (0.25, 0.25, 0.25)]}
calc = Structure(SI_AVEC, SI_SPECIES).get_calculation("_scratch/si_eigenstates", xc="PW", ngridk=(4, 4, 4))
calc.get_energy()

-578.07279182424

### The LAPW generalized eigenvalue problem
Elk's augmented-plane-wave basis is non-orthogonal, $S_{ij}(\mathbf k)=\langle\phi_i(\mathbf k)|\phi_j(\mathbf k)\rangle\neq\delta_{ij}$, so diagonalizing the Hamiltonian means solving
$$ H(\mathbf k)\,\mathbf c_n=E_n\,S(\mathbf k)\,\mathbf c_n $$
The second-variational step then builds the spin-dependent Hamiltonian in the already-orthonormal basis of first-variational states, so its eigenvectors (`evecsv`) satisfy plain unitarity,
$$ \texttt{evecsv}^\dagger\,\texttt{evecsv}=\mathbb 1 $$

In [2]:
state = calc.get_eigenstates((0.1, 0.2, 0.05))
gram = state.evecsv.conj().T @ state.evecsv
print(np.abs(gram - np.eye(gram.shape[0])).max())

0.0


### Wavefunction overlap between two k-points
$$ O_{ab}(\mathbf k_a,\mathbf k_b)=\langle\psi_a(\mathbf k_a)|\psi_b(\mathbf k_b)\rangle=\int_{\rm cell}\psi_a^*(\mathbf k_a,\mathbf r)\,\psi_b(\mathbf k_b,\mathbf r)\,d^3r $$
the same quantity underlying the Wilson-loop link variable of `05_berry_curvature.ipynb`, evaluated here at two independently-diagonalized, arbitrary $\mathbf k$-points -- $O_{ab}(\mathbf k,\mathbf k)$ must be the identity.

In [3]:
with calc.eigenstate_session() as session: # one warm Elk process for many queries
    self_overlap = session.overlap((0.1, 0.2, 0.05), (0.1, 0.2, 0.05), ist0=1, ist1=4)
    cross_overlap = session.overlap((0.0, 0.0, 0.0), (0.5, 0.0, 0.0), ist0=1, ist1=1)

print(np.round(self_overlap, 4))
print(cross_overlap[0, 0])

[[ 9.9990e-01-0.j      5.0000e-04-0.0004j  4.0000e-04+0.0002j
   1.0000e-04+0.0002j]
 [ 5.0000e-04+0.0004j  1.0008e+00+0.j      3.0000e-04+0.0006j
  -2.0000e-04-0.0002j]
 [ 4.0000e-04-0.0002j  3.0000e-04-0.0006j  9.9990e-01-0.j
  -4.0000e-04+0.001j ]
 [ 1.0000e-04-0.0002j -2.0000e-04+0.0002j -4.0000e-04-0.001j
   9.9900e-01-0.j    ]]
(0.7708653877396657-0.00014263905121159547j)
